In [2]:
# importing the dependencies
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [3]:
# importing the models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [4]:
heart_data = pd.read_csv('/content/heart_disease_data.csv')

In [5]:
heart_data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [6]:
heart_data.shape

(303, 14)

In [7]:
heart_data.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [8]:
heart_data['target'].value_counts()

,count
target,
1,165
0,138


1 --> Defective Heart

0 --> Healthy Heart

Splitting the Features and Target

In [9]:
X = heart_data.drop(columns='target', axis=1)
Y = heart_data['target']

In [10]:
print(X)

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
0     63    1   3       145   233    1        0      150      0      2.3   
1     37    1   2       130   250    0        1      187      0      3.5   
2     41    0   1       130   204    0        0      172      0      1.4   
3     56    1   1       120   236    0        1      178      0      0.8   
4     57    0   0       120   354    0        1      163      1      0.6   
..   ...  ...  ..       ...   ...  ...      ...      ...    ...      ...   
298   57    0   0       140   241    0        1      123      1      0.2   
299   45    1   3       110   264    0        1      132      0      1.2   
300   68    1   0       144   193    1        1      141      0      3.4   
301   57    1   0       130   131    0        1      115      1      1.2   
302   57    0   1       130   236    0        0      174      0      0.0   

     slope  ca  thal  
0        0   0     1  
1        0   0     2  
2        2   0    

In [11]:
print(Y)

0      1
1      1
2      1
3      1
4      1
      ..
298    0
299    0
300    0
301    0
302    0
Name: target, Length: 303, dtype: int64


In [12]:
X = np.asarray(X)
Y = np.asarray(Y)

# **Model Selection**

## Comparing the models with default hyperparameter values using Cross Validation

In [15]:
# list of models
models = [LogisticRegression(max_iter=1000), SVC(kernel='linear'), KNeighborsClassifier(), RandomForestClassifier(random_state=0)]

In [16]:
def compare_models_cross_validation():

    for model in models:
        cv_score = cross_val_score(model, X, Y, cv=5)
        mean_accuracy = np.mean(cv_score)*100
        mean_accuracy = round(mean_accuracy, 2)

        print(f'Cross Validation accuracy for {model} is: {cv_score}')
        print(f'Accuracy score of {model} is: {mean_accuracy}%')
        print('------------------------------------------------')

In [17]:
compare_models_cross_validation()

Cross Validation accuracy for LogisticRegression(max_iter=1000) is: [0.80327869 0.86885246 0.85245902 0.86666667 0.75      ]
Accuracy score of LogisticRegression(max_iter=1000) is: 82.83%
------------------------------------------------
Cross Validation accuracy for SVC(kernel='linear') is: [0.81967213 0.8852459  0.80327869 0.86666667 0.76666667]
Accuracy score of SVC(kernel='linear') is: 82.83%
------------------------------------------------
Cross Validation accuracy for KNeighborsClassifier() is: [0.60655738 0.6557377  0.57377049 0.73333333 0.65      ]
Accuracy score of KNeighborsClassifier() is: 64.39%
------------------------------------------------
Cross Validation accuracy for RandomForestClassifier(random_state=0) is: [0.85245902 0.90163934 0.81967213 0.81666667 0.8       ]
Accuracy score of RandomForestClassifier(random_state=0) is: 83.81%
------------------------------------------------


## Comparing the models with different Hyperparameter values using GridSearchCV

In [20]:
# list of models
models = [LogisticRegression(max_iter=10000), SVC(), KNeighborsClassifier(), RandomForestClassifier(random_state=0)]

In [21]:
model_hyperparameters = {
    
    'log_reg_hyperparameters': {
        'C' : [1,5,10,20]
    },

    'svc_hyperparameters': {
        'kernel' : ['linear','poly','rbf','sigmoid'],
        'C' : [1,5,10,20]
    },

    'KNN_hyperparameters' : {
        'n_neighbors' : [3,5,10]
    },

    'random_forest_hyperparameters' : {        
        'n_estimators' : [10, 20, 50, 100]
    }
}

In [22]:
model_keys = list(model_hyperparameters.keys())
print(model_keys)

['log_reg_hyperparameters', 'svc_hyperparameters', 'KNN_hyperparameters', 'random_forest_hyperparameters']


In [28]:
# grid search cv
def ModelSelection(models, model_hyperparameters):

    results = []
    i = 0

    for model in models:
        key = model_keys[i]
        params = model_hyperparameters[key]
        i += 1
        print(model, 'params: ', params)

        classifier = GridSearchCV(model, params, cv=5)
        classifier.fit(X, Y)

        results.append({
            'model': model,
            'best_score': classifier.best_score_,
            'best_params': classifier.best_params_
        })

    result_df = pd.DataFrame(results, columns=['model', 'best_score', 'best_params'])
    return result_df


In [29]:
ModelSelection(models, model_hyperparameters)

LogisticRegression(max_iter=10000) params:  {'C': [1, 5, 10, 20]}
SVC() params:  {'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], 'C': [1, 5, 10, 20]}
KNeighborsClassifier() params:  {'n_neighbors': [3, 5, 10]}
RandomForestClassifier(random_state=0) params:  {'n_estimators': [10, 20, 50, 100]}


,model,best_score,best_params
0,LogisticRegression(max_iter=10000),0.831585,{'C': 5}
1,SVC(),0.828306,"{'C': 1, 'kernel': 'linear'}"
2,KNeighborsClassifier(),0.643880,{'n_neighbors': 5}
3,RandomForestClassifier(random_state=0),0.838087,{'n_estimators': 100}
